# Поиск больших расхождений на новых текстах

Проверяем гипотезу: **случайные измерения для карты доменов, KRR или GP для поиска
следующих текстов с высокой JSD**. Настройки предикторов прежние, данные новые.

600 текстов из шести новых категорий MMLU,20 seeds. Одинаковый старт192 и следующий
пакет60 у всех методов. Для независимой проверки качества поиска измеряем все600.
Эти600 — фактический расход проверки;252 — сравниваемый рабочий бюджет метода.

В Colab выбрать **Среда выполнения → Сменить среду выполнения → T4 GPU**.
Выполнить по порядку; при запросе загрузить **qwen_srs_search_fresh_inputs.zip**.
Подготовка скачивает токенизаторы/веса и исключает старые тексты, поэтому первый
запуск дольше повторного. Готовых результатов нового эксперимента здесь нет.


In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, time, zipfile
from google.colab import files

ROOT = Path('/content/hybrid_fresh_confirmation_20261008')
DEPS = Path('/content/qwen_fresh_confirmation_deps_20261008')
INPUT_NAME = 'qwen_srs_search_fresh_inputs.zip'
INPUT_SHA256 = 'c6b1176d6183f5aca6cd01eed1f7eb0599d34f0a8d4836c232bd8ef136ccd55e'
archive_path = Path('/content') / INPUT_NAME
if not archive_path.exists() or hashlib.sha256(archive_path.read_bytes()).hexdigest() != INPUT_SHA256:
    uploaded = files.upload()
    matching = [blob for blob in uploaded.values() if hashlib.sha256(blob).hexdigest() == INPUT_SHA256]
    if not matching:
        raise ValueError('Нужен входной ZIP из этой же пары файлов')
    archive_path.write_bytes(matching[0])  # Downloads may add "(1)" to the filename.
if hashlib.sha256(archive_path.read_bytes()).hexdigest() != INPUT_SHA256:
    raise ValueError('Архив отличается от подготовленного: используйте свежую пару файлов')
ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(archive_path) as archive:
    archive.extractall(ROOT)
study = json.loads((ROOT / 'study.json').read_text(encoding='utf-8'))
print('Загружены исходники и история:', study['history_rows'], 'текстов. Новых JSD в архиве нет.')


## Среда

Зависимости ставим в отдельную папку. Этапы запускаются отдельными Python-процессами:
так уже загруженные в Colab библиотеки не меняют версии расчёта. Torch и CUDA берём
из текущего Colab; доступность GPU проверяем до скачивания моделей.


In [ ]:
DEPS.mkdir(exist_ok=True)
pin_marker = DEPS / 'installed_pins.json'
pins = study['runtime_pins']
if not pin_marker.exists() or json.loads(pin_marker.read_text()) != pins:
    packages = [f'{name}=={version}' for name, version in pins.items()]
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
                    '--target', str(DEPS), *packages], check=True)
    pin_marker.write_text(json.dumps(pins))
print('Зависимости готовы.')


In [ ]:
def run_stage(stage):
    """Real exit code and full log; only progress is printed in the notebook."""
    logs = ROOT / 'logs'
    logs.mkdir(exist_ok=True)
    env = os.environ.copy()
    env.update(PYTHONPATH=str(DEPS) + os.pathsep + str(ROOT),
               PYTHONUNBUFFERED='1', HF_HUB_DISABLE_PROGRESS_BARS='1',
               TRANSFORMERS_VERBOSITY='error', TOKENIZERS_PARALLELISM='false',
               MPLBACKEND='Agg')
    log_path = logs / f'{stage}.log'
    started = time.perf_counter()
    progress = ('RUNTIME', 'HISTORY', 'DOMAIN', 'MODEL_READY', 'SETUP_',
                'MEASUREMENTS_', 'JSD', 'SELECTION_', 'ORACLE_',
                'EVALUATION', 'RESULT_', 'Reused', 'COMPLETED', 'FRESH_')
    with log_path.open('w', encoding='utf-8') as log:
        process = subprocess.Popen([sys.executable, '-u', str(ROOT / 'pipeline.py'), stage],
                                   cwd=ROOT, env=env, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True,
                                   encoding='utf-8', errors='replace')
        for line in process.stdout:
            log.write(line)
            log.flush()
            if line.startswith(progress):
                print(line.rstrip(), flush=True)
        exit_code = process.wait()
    receipt = dict(stage=stage, exit_code=exit_code,
                   wall_seconds=time.perf_counter()-started, log=log_path.name)
    (logs / f'{stage}_receipt.json').write_text(json.dumps(receipt, indent=2))
    print(stage, 'exit code:', exit_code, 'время:', round(receipt['wall_seconds'], 1), 'с')
    if exit_code:
        print('\n'.join(log_path.read_text(encoding='utf-8').splitlines()[-45:]))
        raise RuntimeError(f'Этап {stage} завершился с ошибкой; полный лог: {log_path}')

run_stage('check')


## Заранее заданный критерий

JSD — различие распределений следующего токена двух Qwen, усреднённое по тексту.
Высокая JSD означает сильное расхождение; правильные ответы MMLU здесь не используются.

На каждом seed оставляем68 неизвестных текстов в домене. Метод выбирает10 из них.
**Recall** — сколько из настоящей десятки с наибольшей JSD удалось найти, делённое на10.
Среднее по шести доменам — итоговая метрика. Random в среднем находит10/68 =14,71%.

Первый вопрос: превышает ли KRR эту случайную планку? Второй: насколько он уступает GP?
Допустимую потерю заранее задаём как **1/60 recall =1,67 процентного пункта**: один
найденный текст из60. Если интервал не позволяет сделать вывод, так и пишем.
Все интервалы относятся к случайным планам на этом пуле; это не20 независимых датасетов.


In [ ]:
print('Teacher:', study['models']['teacher']['id'])
print('Student:', study['models']['student']['id'])
print('Encoder:', study['models']['encoder']['id'])
print('Seeds:', study['seeds'])
print('Приоритет доменов:', study['domain_priority'])
print('Допустимая потеря KRR:', round(100 * study['noninferiority_margin'], 3), 'п.п.')
print('SHA протокола:', hashlib.sha256((ROOT / 'study.json').read_bytes()).hexdigest())


## Новые данные и BGE

Берём первые шесть категорий из заданного списка, где после исключения старых текстов
остаётся хотя бы100 примеров. Это решение принимается **до измерения JSD**.
Исключаем повторы исходных текстов, одинаковые видимые Qwen-префиксы и близкие копии
старых текстов. Свежесть проверяется относительно приложенной истории, а не pretraining.

Выбираем общий raw prefix: именно этот фрагмент исходной строки видят и Qwen, и BGE.
Ограничение —256 токенов каждого токенизатора. BGE даёт последний CLS-вектор с L2
нормализацией. Сохраняем600 строк, признаки и20 случайных планов прежде любых JSD.


In [ ]:
run_stage('prepare')


## Измерения случайного старта

Для каждого seed случайно выбираем32 текста на домен, всего192. Старт общий для всех
предикторов. Реальные Qwen-вычисления объединяем в общий cache: объединение20 стартов
может охватить почти все600 текстов. Но при обучении каждый seed получает **только свои192
метки**. Полная таблица JSD ещё не собирается и в предиктор не передаётся.

На каждом тексте: forward teacher и student → полная словарная JSD в fp32 → среднее по
всем видимым позициям, включая последнюю. Измерения сохраняются сразу, по одному тексту.


In [ ]:
run_stage('initial')


## Прогнозы и следующий пакет

Один общий GP или KRR обучается на192 метках всех шести доменов. Предсказываем JSD
оставшихся текстов и берём10 с наибольшим прогнозом в каждом домене. Дополнительно
оставлены BayesianRidge, средние четырёх кластеров в домене и среднее домена.
В GP, KRR и остальных предикторах настройки прежнего эксперимента не менялись.
Random берёт10 текстов без возвращения из того же остатка.

Сохраняем прогнозы и выборы всех методов. Полный oracle открываем только после этого.
Исходники расчёта доступны в панели Files: pipeline.py, experiment.py, evaluation.py.


In [ ]:
run_stage('select')


## Полная проверка

Досчитываем ещё не измеренные тексты. После сохранения всех выборов собираем полный
oracle — реальные JSD всех600 текстов. По нему проверяем recall, MAE/R² и карту доменов.

θ — доля текстов с JSD≤0,05. Для гибридной схемы оцениваем её по случайным32 текстам
домена.60 текстов, специально выбранных за высокую JSD, в эту простую долю не добавляем.
Для полностью случайного метода дополнительно считаем долю по42 текстам на домен.


In [ ]:
run_stage('remaining')
run_stage('oracle')
run_stage('evaluate')


## Результаты

Отчёт создаётся из текущего запуска. В нём отдельно указаны качество поиска, оценка θ,
порядок доменов и расходы: BGE, Qwen, токены, fit/predict. Random не требует BGE.
20 fits каждого предиктора и oracle600 — расходы исследования, а не одной рабочей выдачи.
Сильные расхождения пока не означают доказанную пользу для обучения student.


In [ ]:
from IPython.display import display, Markdown
display(Markdown((ROOT / 'results/РЕЗУЛЬТАТЫ.md').read_text(encoding='utf-8')))


In [ ]:
run_stage('package')
files.download('/content/qwen_srs_search_fresh_results.zip')


Передайте **qwen_srs_search_fresh_results.zip**. Он содержит текущие измерения,
выборы до проверки, полный протокол, исходники, отчёт и логи с exit codes.
При обрыве в том же сеансе повторите упавший этап: cache сохраняет уже готовые измерения.
Не меняйте настройки по ходу этого подтверждающего запуска.
